# 02 – DQ Rule Onboarding
Add, change or deactivate rules here, then run the **MERGE** cell.

**How to convert a functional-team query (Option 2):**
- `base_sql` = their query **without** the failure filter → this is the denominator. Must return `record_key`
  plus every column the condition uses.
- `failure_condition` = only the failure filter (what was in their WHERE for "bad" data).
- Complex rules that are not a simple filter (duplicates, "has no active site"...): leave `failure_condition = None`
  and put the full failing query in `failure_sql` (must also return `record_key`).
- Queries must be **Spark SQL** (not T-SQL). Refer to source tables as `{SRC}.table_name`.

**Operating unit (OU):**
- `entity_grain = 'CUSTOMER'` (record_key = cust_account_id): nothing to add – the runner assigns the customer to
  **every** OU where it has a site.
- Site-level or other rules: return an `org_id` column in `base_sql` (and in `failure_sql` for complex rules).
- No OU found → `org_id = -1` ("Unassigned").

**Record attributes (optional, for record-level filters):** if the failing rows contain any of
`account_number, party_number, cust_account_id, cust_acct_site_id`, they are stored with the failed records.
For condition rules, add them to `base_sql`'s SELECT.

**Behaviour:** `rule_id` is the key. Changing SQL bumps `rule_version`. Rules not in the list are left untouched.
To stop a rule, set `is_active = False` and re-run the MERGE. Never reuse a `rule_id`.

In [ ]:
%run 00_dq_config

In [ ]:
# Step 1: define rules  (EXAMPLES - replace with your real rules)
rules = [
    dict(
        rule_id=1,
        rule_name="Customer missing tax reference",
        rule_description="Active customer accounts whose party has no tax reference",
        dimension_name="Completeness",
        entity_grain="CUSTOMER",
        base_sql=f"""
            SELECT ca.cust_account_id AS record_key,
                   p.tax_reference
            FROM {SRC}.hz_cust_accounts ca
            JOIN {SRC}.hz_parties p ON p.party_id = ca.party_id
            WHERE ca.status = 'A'
        """,
        failure_condition="tax_reference IS NULL OR trim(tax_reference) = ''",
        failure_sql=None,
        severity="High",
        business_owner="Customer MDM team",
        is_active=True,
    ),
    dict(
        rule_id=2,
        rule_name="Duplicate customer name",
        rule_description="Active customer accounts sharing the same party name with another active account",
        dimension_name="Uniqueness",
        entity_grain="CUSTOMER",
        base_sql=f"""
            SELECT ca.cust_account_id AS record_key
            FROM {SRC}.hz_cust_accounts ca
            WHERE ca.status = 'A'
        """,
        failure_condition=None,           # complex rule -> use failure_sql
        failure_sql=f"""
            WITH n AS (
                SELECT ca.cust_account_id, upper(trim(p.party_name)) AS nm
                FROM {SRC}.hz_cust_accounts ca
                JOIN {SRC}.hz_parties p ON p.party_id = ca.party_id
                WHERE ca.status = 'A'
            )
            SELECT cust_account_id AS record_key
            FROM n
            WHERE nm IN (SELECT nm FROM n GROUP BY nm HAVING count(*) > 1)
        """,
        severity="Medium",
        business_owner="Customer MDM team",
        is_active=True,
    ),
]

In [ ]:
# Step 2 (optional but recommended): test that each rule's SQL runs before saving it
for r in rules:
    try:
        base = spark.sql(r["base_sql"])
        assert "record_key" in base.columns, "base_sql must return a column named record_key"
        if r["failure_condition"]:
            base.filter(r["failure_condition"]).limit(1).collect()
        else:
            f = spark.sql(r["failure_sql"])
            assert "record_key" in f.columns, "failure_sql must return a column named record_key"
            f.limit(1).collect()
        print(f"OK     rule {r['rule_id']}: {r['rule_name']}")
    except Exception as e:
        print(f"ERROR  rule {r['rule_id']}: {r['rule_name']} -> {str(e)[:300]}")

In [ ]:
# Step 3: MERGE rules into the catalog
cols = ["rule_id", "rule_name", "rule_description", "dimension_name", "entity_grain",
        "base_sql", "failure_condition", "failure_sql", "severity", "business_owner", "is_active"]

src_schema = spark.table(T_CATALOG).select(*cols).schema
src_df = spark.createDataFrame([tuple(r[c] for c in cols) for r in rules], src_schema)
src_df.createOrReplaceTempView("src_rules")

spark.sql(f"""
MERGE INTO {T_CATALOG} t
USING src_rules s
ON t.rule_id = s.rule_id

WHEN MATCHED AND (
       t.base_sql <> s.base_sql
    OR coalesce(t.failure_condition, '') <> coalesce(s.failure_condition, '')
    OR coalesce(t.failure_sql, '')       <> coalesce(s.failure_sql, '')
) THEN UPDATE SET
    t.rule_name         = s.rule_name,
    t.rule_description  = s.rule_description,
    t.dimension_name    = s.dimension_name,
    t.entity_grain      = s.entity_grain,
    t.base_sql          = s.base_sql,
    t.failure_condition = s.failure_condition,
    t.failure_sql       = s.failure_sql,
    t.severity          = s.severity,
    t.business_owner    = s.business_owner,
    t.is_active         = s.is_active,
    t.rule_version      = t.rule_version + 1,
    t.modified_ts       = current_timestamp()

WHEN MATCHED THEN UPDATE SET
    t.rule_name         = s.rule_name,
    t.rule_description  = s.rule_description,
    t.dimension_name    = s.dimension_name,
    t.entity_grain      = s.entity_grain,
    t.severity          = s.severity,
    t.business_owner    = s.business_owner,
    t.is_active         = s.is_active,
    t.modified_ts       = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
    rule_id, rule_name, rule_description, dimension_name, entity_grain,
    base_sql, failure_condition, failure_sql, severity, business_owner,
    is_active, rule_version, created_ts, modified_ts
) VALUES (
    s.rule_id, s.rule_name, s.rule_description, s.dimension_name, s.entity_grain,
    s.base_sql, s.failure_condition, s.failure_sql, s.severity, s.business_owner,
    s.is_active, 1, current_timestamp(), current_timestamp()
)
""")

display(spark.table(T_CATALOG)
             .select("rule_id", "rule_name", "dimension_name", "is_active", "rule_version", "modified_ts")
             .orderBy("rule_id"))

In [ ]:
# Step 4 (optional): manual link per rule -> dq_rule_export_url
# Add  url="https://..."  to a rule's dict in Step 1 and run this cell. Rules without "url" are left untouched.
from delta.tables import DeltaTable

T_RULE_URL = globals().get("T_RULE_URL", f"{DQ}.dq_rule_export_url")
spark.sql(f"CREATE TABLE IF NOT EXISTS {T_RULE_URL} (rule_id INT, url STRING, updated_ts TIMESTAMP) USING DELTA")

url_rows = [(int(r["rule_id"]), r["url"]) for r in rules if r.get("url")]

if url_rows:
    src = (spark.createDataFrame(url_rows, "rule_id INT, url STRING")
                .withColumn("updated_ts", F.current_timestamp()))
    (DeltaTable.forName(spark, T_RULE_URL).alias("t")
        .merge(src.alias("s"), "t.rule_id = s.rule_id")
        .whenMatchedUpdate(condition="t.url <> s.url OR t.url IS NULL",
                           set={"url": "s.url", "updated_ts": "s.updated_ts"})
        .whenNotMatchedInsertAll()
        .execute())
    print(f"{len(url_rows)} rule link(s) merged")
else:
    print("No rule has a url - nothing to merge")

display(spark.table(T_RULE_URL).orderBy("rule_id"))